In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# CROP CALENDAR DATASET - PRODUCTION QUALITY CLEANING
# ============================================================

INPUT_FILE = "sacks_crop_calendar_india.csv"
OUTPUT_FILE = "sacks_crop_calendar_india_clean.csv"

# Create output directory
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("CROP CALENDAR DATASET CLEANING")
print("=" * 60)

print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

required_columns = [
    "country",
    "latitude",
    "longitude",
    "crop",
    "planting_start_day",
    "planting_end_day",
    "harvest_start_day",
    "harvest_end_day",
    "planting_start_month",
    "planting_end_month",
    "harvest_start_month",
    "harvest_end_month"
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# Keep expected columns
df = df[required_columns].copy()

# ------------------------------------------------------------
# 3. Clean text columns
# ------------------------------------------------------------

df["country"] = (
    df["country"]
    .astype("string")
    .str.strip()
)

df["crop"] = (
    df["crop"]
    .astype("string")
    .str.strip()
)

# Standardize country names
df["country"] = df["country"].replace({
    "IND": "India",
    "INDIA": "India",
    "india": "India"
})

# ------------------------------------------------------------
# 4. Convert numeric columns
# ------------------------------------------------------------

numeric_columns = [
    "latitude",
    "longitude",
    "planting_start_day",
    "planting_end_day",
    "harvest_start_day",
    "harvest_end_day",
    "planting_start_month",
    "planting_end_month",
    "harvest_start_month",
    "harvest_end_month"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )

# ------------------------------------------------------------
# 5. Validate latitude
# ------------------------------------------------------------

invalid_latitude = ~df["latitude"].between(
    -90,
    90
)

df.loc[
    invalid_latitude,
    "latitude"
] = pd.NA

# ------------------------------------------------------------
# 6. Validate longitude
# ------------------------------------------------------------

invalid_longitude = ~df["longitude"].between(
    -180,
    180
)

df.loc[
    invalid_longitude,
    "longitude"
] = pd.NA

# ------------------------------------------------------------
# 7. Validate day-of-year values
# ------------------------------------------------------------

day_columns = [
    "planting_start_day",
    "planting_end_day",
    "harvest_start_day",
    "harvest_end_day"
]

invalid_day_count = 0

for column in day_columns:

    invalid_days = ~df[column].between(
        1,
        365
    )

    invalid_day_count += invalid_days.sum()

    df.loc[
        invalid_days,
        column
    ] = pd.NA

# ------------------------------------------------------------
# 8. Validate month values
# ------------------------------------------------------------

month_columns = [
    "planting_start_month",
    "planting_end_month",
    "harvest_start_month",
    "harvest_end_month"
]

invalid_month_count = 0

for column in month_columns:

    invalid_months = ~df[column].between(
        1,
        12
    )

    invalid_month_count += invalid_months.sum()

    df.loc[
        invalid_months,
        column
    ] = pd.NA

# ------------------------------------------------------------
# 9. Remove exact duplicates
# ------------------------------------------------------------

rows_before_duplicates = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

duplicates_removed = (
    rows_before_duplicates - len(df)
)

# ------------------------------------------------------------
# 10. IMPORTANT:
# Do NOT remove start > end calendar ranges
# ------------------------------------------------------------

# Some agricultural seasons cross the end of the year.
#
# Example:
#
# Planting:
# Day 246 → Day 63
#
# means:
#
# Day 246 → Day 365 → Day 1 → Day 63
#
# Therefore these are retained as valid seasonal ranges.

planting_cross_year = (
    df["planting_start_day"]
    > df["planting_end_day"]
)

harvest_cross_year = (
    df["harvest_start_day"]
    > df["harvest_end_day"]
)

# ------------------------------------------------------------
# 11. Convert calendar fields to integer nullable type
# ------------------------------------------------------------

for column in day_columns + month_columns:
    df[column] = df[column].astype("Int64")

# ------------------------------------------------------------
# 12. Final validation report
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(
    "Rows before cleaning:",
    rows_before_duplicates
)

print(
    "Rows after cleaning:",
    len(df)
)

print(
    "Duplicate rows removed:",
    duplicates_removed
)

print(
    "Invalid day values:",
    invalid_day_count
)

print(
    "Invalid month values:",
    invalid_month_count
)

print(
    "Planting seasons crossing year:",
    planting_cross_year.sum()
)

print(
    "Harvest seasons crossing year:",
    harvest_cross_year.sum()
)

print("\nMissing values:")

print(
    df.isna().sum()
)

print("\nData types:")

print(
    df.dtypes
)

# ------------------------------------------------------------
# 13. Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Cleaned file saved to:",
    OUTPUT_FILE
)

CROP CALENDAR DATASET CLEANING
Original shape: (3666, 12)

CLEANING SUMMARY
Rows before cleaning: 3666
Rows after cleaning: 3666
Duplicate rows removed: 0
Invalid day values: 0
Invalid month values: 0
Planting seasons crossing year: 27
Harvest seasons crossing year: 138

Missing values:
country                 0
latitude                0
longitude               0
crop                    0
planting_start_day      0
planting_end_day        0
harvest_start_day       0
harvest_end_day         0
planting_start_month    0
planting_end_month      0
harvest_start_month     0
harvest_end_month       0
dtype: int64

Data types:
country                 string[python]
latitude                       float64
longitude                      float64
crop                    string[python]
planting_start_day               Int64
planting_end_day                 Int64
harvest_start_day                Int64
harvest_end_day                  Int64
planting_start_month             Int64
planting_end_month     